# Credit Score Classification

# 01 Data Preparation

## Import Library

In [1]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

import re
import time
import random
import tempfile
from tqdm.notebook import tqdm

import gc

## Import Dataset

In [2]:
df_origin_train = pd.read_csv('dataset/train.csv')
df_train = df_origin_train.copy()

C:\Users\watta\AppData\Local\Temp\ipykernel_17964\431262077.py:1: DtypeWarning: Columns (26) have mixed types. Specify dtype option on import or set low_memory=False.
  df_origin_train = pd.read_csv('dataset/train.csv')


## Check Dataset

In [3]:
df_train

,ID,Customer_ID,Month,Name,Age,SSN,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,...,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,Credit_Score
0,0x1602,CUS_0xd40,January,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,1824.843333,3,...,_,809.98,26.822620,22 Years and 1 Months,No,49.574949,80.41529543900253,High_spent_Small_value_payments,312.49408867943663,Good
1,0x1603,CUS_0xd40,February,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,NaN,3,...,Good,809.98,31.944960,NaN,No,49.574949,118.28022162236736,Low_spent_Large_value_payments,284.62916249607184,Good
2,0x1604,CUS_0xd40,March,Aaron Maashoh,-500,821-00-0265,Scientist,19114.12,NaN,3,...,Good,809.98,28.609352,22 Years and 3 Months,No,49.574949,81.699521264648,Low_spent_Medium_value_payments,331.2098628537912,Good
3,0x1605,CUS_0xd40,April,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,NaN,3,...,Good,809.98,31.377862,22 Years and 4 Months,No,49.574949,199.4580743910713,Low_spent_Small_value_payments,223.45130972736786,Good
4,0x1606,CUS_0xd40,May,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,1824.843333,3,...,Good,809.98,24.797347,22 Years and 5 Months,No,49.574949,41.420153086217326,High_spent_Medium_value_payments,341.48923103222177,Good
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,0x25fe9,CUS_0x942c,April,Nicks,25,078-73-5990,Mechanic,39628.99,3359.415833,4,...,_,502.38,34.663572,31 Years and 6 Months,No,35.104023,60.97133255718485,High_spent_Large_value_payments,479.866228,Poor
99996,0x25fea,CUS_0x942c,May,Nicks,25,078-73-5990,Mechanic,39628.99,3359.415833,4,...,_,502.38,40.565631,31 Years and 7 Months,No,35.104023,54.18595028760385,High_spent_Medium_value_payments,496.65161,Poor
99997,0x25feb,CUS_0x942c,June,Nicks,25,078-73-5990,Mechanic,39628.99,3359.415833,4,...,Good,502.38,41.255522,31 Years and 8 Months,No,35.104023,24.02847744864441,High_spent_Large_value_payments,516.809083,Poor
99998,0x25fec,CUS_0x942c,July,Nicks,25,078-73-5990,Mechanic,39628.99,3359.415833,4,...,Good,502.38,33.638208,31 Years and 9 Months,No,35.104023,251.67258219721603,Low_spent_Large_value_payments,319.164979,Standard


## Clear Strange Value

In [4]:
def text_cleaning(data):
    if data is np.NaN or not isinstance(data, str):
        return data
    else:
        return str(data).strip('_ ,"')

In [5]:
df_train = df_train.applymap(text_cleaning).replace(['', 'nan', '!@9#%8', '#F%$D@*&8'], np.NaN)

In [6]:
df_train

,ID,Customer_ID,Month,Name,Age,SSN,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,...,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,Credit_Score
0,0x1602,CUS_0xd40,January,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,1824.843333,3,...,NaN,809.98,26.822620,22 Years and 1 Months,No,49.574949,80.41529543900253,High_spent_Small_value_payments,312.49408867943663,Good
1,0x1603,CUS_0xd40,February,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,NaN,3,...,Good,809.98,31.944960,NaN,No,49.574949,118.28022162236736,Low_spent_Large_value_payments,284.62916249607184,Good
2,0x1604,CUS_0xd40,March,Aaron Maashoh,-500,821-00-0265,Scientist,19114.12,NaN,3,...,Good,809.98,28.609352,22 Years and 3 Months,No,49.574949,81.699521264648,Low_spent_Medium_value_payments,331.2098628537912,Good
3,0x1605,CUS_0xd40,April,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,NaN,3,...,Good,809.98,31.377862,22 Years and 4 Months,No,49.574949,199.4580743910713,Low_spent_Small_value_payments,223.45130972736786,Good
4,0x1606,CUS_0xd40,May,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,1824.843333,3,...,Good,809.98,24.797347,22 Years and 5 Months,No,49.574949,41.420153086217326,High_spent_Medium_value_payments,341.48923103222177,Good
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,0x25fe9,CUS_0x942c,April,Nicks,25,078-73-5990,Mechanic,39628.99,3359.415833,4,...,NaN,502.38,34.663572,31 Years and 6 Months,No,35.104023,60.97133255718485,High_spent_Large_value_payments,479.866228,Poor
99996,0x25fea,CUS_0x942c,May,Nicks,25,078-73-5990,Mechanic,39628.99,3359.415833,4,...,NaN,502.38,40.565631,31 Years and 7 Months,No,35.104023,54.18595028760385,High_spent_Medium_value_payments,496.65161,Poor
99997,0x25feb,CUS_0x942c,June,Nicks,25,078-73-5990,Mechanic,39628.99,3359.415833,4,...,Good,502.38,41.255522,31 Years and 8 Months,No,35.104023,24.02847744864441,High_spent_Large_value_payments,516.809083,Poor
99998,0x25fec,CUS_0x942c,July,Nicks,25,078-73-5990,Mechanic,39628.99,3359.415833,4,...,Good,502.38,33.638208,31 Years and 9 Months,No,35.104023,251.67258219721603,Low_spent_Large_value_payments,319.164979,Standard


## Check For NaN

In [7]:
df_train.isna().sum()

ID                              0
Customer_ID                     0
Month                           0
Name                         9985
Age                             0
SSN                          5572
Occupation                   7062
Annual_Income                   0
Monthly_Inhand_Salary       15002
Num_Bank_Accounts               0
Num_Credit_Card                 0
Interest_Rate                   0
Num_of_Loan                     0
Type_of_Loan                11408
Delay_from_due_date             0
Num_of_Delayed_Payment       7002
Changed_Credit_Limit         2091
Num_Credit_Inquiries         1965
Credit_Mix                  20195
Outstanding_Debt                0
Credit_Utilization_Ratio        0
Credit_History_Age           9030
Payment_of_Min_Amount           0
Total_EMI_per_month             0
Amount_invested_monthly      4479
Payment_Behaviour            7600
Monthly_Balance              1200
Credit_Score                    0
dtype: int64

## Check For Statistic

In [8]:
pd.set_option('display.float_format', '{:.2f}'.format)

In [9]:
df_train.describe().T

,count,mean,std,min,25%,50%,75%,max
Monthly_Inhand_Salary,84998.00,4194.17,3183.69,303.65,1625.57,3093.75,5957.45,15204.63
Num_Bank_Accounts,100000.00,17.09,117.40,-1.00,3.00,6.00,7.00,1798.00
Num_Credit_Card,100000.00,22.47,129.06,0.00,4.00,5.00,7.00,1499.00
Interest_Rate,100000.00,72.47,466.42,1.00,8.00,13.00,20.00,5797.00
Delay_from_due_date,100000.00,21.07,14.86,-5.00,10.00,18.00,28.00,67.00
Num_Credit_Inquiries,98035.00,27.75,193.18,0.00,3.00,6.00,9.00,2597.00
Credit_Utilization_Ratio,100000.00,32.29,5.12,20.00,28.05,32.31,36.50,50.00
Total_EMI_per_month,100000.00,1403.12,8306.04,0.00,30.31,69.25,161.22,82331.00


## Fix Parameter Type

In [10]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 28 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   ID                        100000 non-null  object 
 1   Customer_ID               100000 non-null  object 
 2   Month                     100000 non-null  object 
 3   Name                      90015 non-null   object 
 4   Age                       100000 non-null  object 
 5   SSN                       94428 non-null   object 
 6   Occupation                92938 non-null   object 
 7   Annual_Income             100000 non-null  object 
 8   Monthly_Inhand_Salary     84998 non-null   float64
 9   Num_Bank_Accounts         100000 non-null  int64  
 10  Num_Credit_Card           100000 non-null  int64  
 11  Interest_Rate             100000 non-null  int64  
 12  Num_of_Loan               100000 non-null  object 
 13  Type_of_Loan              88592 non-null   ob

In [11]:
df_train['ID']                      = df_train.ID.apply(lambda x: int(x, 16))
df_train['Customer_ID']             = df_train.Customer_ID.apply(lambda x: int(x[4:], 16))
df_train['Month']                   = pd.to_datetime(df_train.Month, format='%B').dt.month
df_train['Age']                     = df_train.Age.astype(int) 
df_train['SSN']                     = df_train.SSN.apply(lambda x: x if x is np.NaN else int(str(x).replace('-', ''))).astype(float)
df_train['Annual_Income']           = df_train.Annual_Income.astype(float)
df_train['Num_of_Loan']             = df_train.Num_of_Loan.astype(int) 
df_train['Num_of_Delayed_Payment']  = df_train.Num_of_Delayed_Payment.astype(float)
df_train['Changed_Credit_Limit']    = df_train.Changed_Credit_Limit.astype(float)
df_train['Outstanding_Debt']        = df_train.Outstanding_Debt.astype(float)
df_train['Amount_invested_monthly'] = df_train.Amount_invested_monthly.astype(float)
df_train['Monthly_Balance']         = df_train.Monthly_Balance.astype(float)

In [12]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 28 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   ID                        100000 non-null  int64  
 1   Customer_ID               100000 non-null  int64  
 2   Month                     100000 non-null  int32  
 3   Name                      90015 non-null   object 
 4   Age                       100000 non-null  int32  
 5   SSN                       94428 non-null   float64
 6   Occupation                92938 non-null   object 
 7   Annual_Income             100000 non-null  float64
 8   Monthly_Inhand_Salary     84998 non-null   float64
 9   Num_Bank_Accounts         100000 non-null  int64  
 10  Num_Credit_Card           100000 non-null  int64  
 11  Interest_Rate             100000 non-null  int64  
 12  Num_of_Loan               100000 non-null  int32  
 13  Type_of_Loan              88592 non-null   ob

## Credit_History_Age

In [13]:
def Month_Converter(x):
    if pd.notnull(x):
        num1 = int(x.split(' ')[0])
        num2 = int(x.split(' ')[3])
      
        return (num1*12)+num2
    else:
        return x

In [14]:
df_train['Credit_History_Age'] = df_train.Credit_History_Age.apply(lambda x: Month_Converter(x)).astype(float)

In [15]:
df_train.groupby('Customer_ID')['Credit_History_Age'].apply(list)

Customer_ID
1006     [182.0, 183.0, 184.0, 185.0, 186.0, 187.0, 188...
1007     [346.0, 347.0, 348.0, 349.0, 350.0, nan, 352.0...
1008     [292.0, 293.0, 294.0, nan, 296.0, 297.0, 298.0...
1009     [331.0, 332.0, 333.0, 334.0, 335.0, 336.0, 337...
1011     [179.0, 180.0, nan, 182.0, 183.0, 184.0, 185.0...
                               ...                        
50984    [157.0, 158.0, 159.0, nan, 161.0, 162.0, nan, ...
50990       [70.0, 71.0, 72.0, nan, nan, 75.0, 76.0, 77.0]
50992    [159.0, 160.0, 161.0, 162.0, 163.0, nan, 165.0...
50996    [206.0, 207.0, 208.0, 209.0, 210.0, 211.0, 212...
50999    [226.0, 227.0, 228.0, 229.0, 230.0, 231.0, 232...
Name: Credit_History_Age, Length: 12500, dtype: object

## Type_of_Loan

In [16]:
def get_Diff_Values_Colum(df_column, diff_value=[], sep=',', replace=''):   
    column = df_column.dropna()
    for i in column:
        if sep not in i and i not in diff_value:
            diff_value.append(i)
        else:
            for data in map(lambda x:x.strip(), re.sub(replace, '', i).split(sep)):
                if not data in diff_value:
                    diff_value.append(data)
    return dict(enumerate(sorted(diff_value)))

In [17]:
df_train['Type_of_Loan'].value_counts(dropna=False).head(20)

Type_of_Loan
NaN                                         11408
Not Specified                                1408
Credit-Builder Loan                          1280
Personal Loan                                1272
Debt Consolidation Loan                      1264
Student Loan                                 1240
Payday Loan                                  1200
Mortgage Loan                                1176
Auto Loan                                    1152
Home Equity Loan                             1136
Personal Loan, and Student Loan               320
Not Specified, and Payday Loan                272
Mortgage Loan, and Home Equity Loan           264
Student Loan, and Payday Loan                 256
Student Loan, and Credit-Builder Loan         248
Credit-Builder Loan, and Not Specified        248
Payday Loan, and Debt Consolidation Loan      240
Payday Loan, and Auto Loan                    240
Mortgage Loan, and Not Specified              232
Payday Loan, and Personal Loan       

In [18]:
df_train.groupby('Customer_ID')['Type_of_Loan'].value_counts(dropna=False)

Customer_ID  Type_of_Loan                                                                                         
1006         Credit-Builder Loan, and Payday Loan                                                                     8
1007         Home Equity Loan, Mortgage Loan, and Student Loan                                                        8
1008         NaN                                                                                                      8
1009         Credit-Builder Loan, Student Loan, Not Specified, and Student Loan                                       8
1011         Personal Loan, Auto Loan, and Auto Loan                                                                  8
                                                                                                                     ..
50984        Home Equity Loan, Mortgage Loan, Payday Loan, Mortgage Loan, Mortgage Loan, and Payday Loan              8
50990        Mortgage Loan, Auto Loan, and Au

In [19]:
df_train.groupby('Customer_ID')['Type_of_Loan'].apply(list)

Customer_ID
1006     [Credit-Builder Loan, and Payday Loan, Credit-...
1007     [Home Equity Loan, Mortgage Loan, and Student ...
1008              [nan, nan, nan, nan, nan, nan, nan, nan]
1009     [Credit-Builder Loan, Student Loan, Not Specif...
1011     [Personal Loan, Auto Loan, and Auto Loan, Pers...
                               ...                        
50984    [Home Equity Loan, Mortgage Loan, Payday Loan,...
50990    [Mortgage Loan, Auto Loan, and Auto Loan, Mort...
50992    [Student Loan, Credit-Builder Loan, Mortgage L...
50996             [nan, nan, nan, nan, nan, nan, nan, nan]
50999    [Credit-Builder Loan, Credit-Builder Loan, Cre...
Name: Type_of_Loan, Length: 12500, dtype: object

In [20]:
get_Diff_Values_Colum(df_train['Type_of_Loan'])

{0: 'Auto Loan',
 1: 'Credit-Builder Loan',
 2: 'Debt Consolidation Loan',
 3: 'Home Equity Loan',
 4: 'Mortgage Loan',
 5: 'Not Specified',
 6: 'Payday Loan',
 7: 'Personal Loan',
 8: 'Student Loan',
 9: 'and Auto Loan',
 10: 'and Credit-Builder Loan',
 11: 'and Debt Consolidation Loan',
 12: 'and Home Equity Loan',
 13: 'and Mortgage Loan',
 14: 'and Not Specified',
 15: 'and Payday Loan',
 16: 'and Personal Loan',
 17: 'and Student Loan'}

## Fill NaN Values of Object Column Function

In [21]:
def Object_NaN_Values_Reassign_Group_Mode(df, groupby, column, inplace=True):      
    def make_NaN_and_fill_mode(df, groupby, column, inplace=True):
        if df[column].isin([None]).sum():
            df[column][df[column].isin([None])] = np.NaN
        
        result = df.groupby(groupby)[column].transform(lambda x: x.fillna(x.mode()[0] if not x.mode().empty else np.NaN))

        if inplace:
            df[column] = result
        else:
            return result
    
    if inplace:  
        if df[column].value_counts(dropna=False).index.isna().sum():
            x = df[column].value_counts(dropna=False).loc[[np.NaN]]
            print(f'\nBefore Assigning: {column}:', f'have {x.values[0]} NaN Values', end='\n')
            
        a = df.groupby(groupby)[column].apply(list) 
        print(f'\nBefore Assigning Example {column}:\n', *a.head().values, sep='\n', end='\n')
        
        make_NaN_and_fill_mode(df, groupby, column, inplace)
        
        if df[column].value_counts(dropna=False).index.isna().sum():
            y = df[column].value_counts(dropna=False).loc[[np.NaN]]
            print(f'\nAfter Assigning: {column}:', f'have {y.values[0]} NaN Values', end='\n')
            
        b = df.groupby(groupby)[column].apply(list)
        print(f'\nAfter Assigning Example {column}:\n', *b.head().values, sep='\n', end='\n')
    else:   
        return make_NaN_and_fill_mode(df, groupby, column, inplace)


## Name

In [22]:
df_train['Name'].value_counts(dropna=False).head()

Name
NaN         9985
Langep        44
Stevex        44
Vaughanl      39
Jessicad      39
Name: count, dtype: int64

In [23]:
Object_NaN_Values_Reassign_Group_Mode(df_train, 'Customer_ID', 'Name')


Before Assigning: Name: have 9985 NaN Values

Before Assigning Example Name:

['Matthias Blamontb', 'Matthias Blamontb', 'Matthias Blamontb', 'Matthias Blamontb', 'Matthias Blamontb', 'Matthias Blamontb', 'Matthias Blamontb', nan]
[nan, 'Soyoung Kimu', 'Soyoung Kimu', 'Soyoung Kimu', 'Soyoung Kimu', nan, nan, 'Soyoung Kimu']
['Koht', 'Koht', 'Koht', 'Koht', 'Koht', 'Koht', 'Koht', nan]
['Edd', 'Edd', 'Edd', 'Edd', 'Edd', 'Edd', 'Edd', 'Edd']
['Terry Wadeu', 'Terry Wadeu', 'Terry Wadeu', 'Terry Wadeu', 'Terry Wadeu', 'Terry Wadeu', 'Terry Wadeu', nan]

After Assigning Example Name:

['Matthias Blamontb', 'Matthias Blamontb', 'Matthias Blamontb', 'Matthias Blamontb', 'Matthias Blamontb', 'Matthias Blamontb', 'Matthias Blamontb', 'Matthias Blamontb']
['Soyoung Kimu', 'Soyoung Kimu', 'Soyoung Kimu', 'Soyoung Kimu', 'Soyoung Kimu', 'Soyoung Kimu', 'Soyoung Kimu', 'Soyoung Kimu']
['Koht', 'Koht', 'Koht', 'Koht', 'Koht', 'Koht', 'Koht', 'Koht']
['Edd', 'Edd', 'Edd', 'Edd', 'Edd', 'Edd', 'Edd

In [24]:
df_train['Name'].value_counts(dropna=False).head()

Name
Jessicad          48
Langep            48
Stevex            48
Vaughanl          40
Ronald Groverk    40
Name: count, dtype: int64

## Occupation

In [25]:
df_train['Occupation'].value_counts(dropna=False)

Occupation
NaN              7062
Lawyer           6575
Architect        6355
Engineer         6350
Scientist        6299
Mechanic         6291
Accountant       6271
Developer        6235
Media_Manager    6232
Teacher          6215
Entrepreneur     6174
Doctor           6087
Journalist       6085
Manager          5973
Musician         5911
Writer           5885
Name: count, dtype: int64

In [26]:
Object_NaN_Values_Reassign_Group_Mode(df_train, 'Customer_ID', 'Occupation')


Before Assigning: Occupation: have 7062 NaN Values

Before Assigning Example Occupation:

['Journalist', 'Journalist', 'Journalist', 'Journalist', 'Journalist', nan, 'Journalist', 'Journalist']
['Manager', 'Manager', nan, 'Manager', 'Manager', 'Manager', 'Manager', 'Manager']
['Developer', 'Developer', 'Developer', 'Developer', 'Developer', 'Developer', 'Developer', 'Developer']
['Accountant', nan, 'Accountant', 'Accountant', 'Accountant', 'Accountant', 'Accountant', 'Accountant']
['Writer', 'Writer', 'Writer', 'Writer', nan, 'Writer', 'Writer', 'Writer']

After Assigning Example Occupation:

['Journalist', 'Journalist', 'Journalist', 'Journalist', 'Journalist', 'Journalist', 'Journalist', 'Journalist']
['Manager', 'Manager', 'Manager', 'Manager', 'Manager', 'Manager', 'Manager', 'Manager']
['Developer', 'Developer', 'Developer', 'Developer', 'Developer', 'Developer', 'Developer', 'Developer']
['Accountant', 'Accountant', 'Accountant', 'Accountant', 'Accountant', 'Accountant', 'Accoun

In [27]:
df_train['Occupation'].value_counts(dropna=False)

Occupation
Lawyer           7096
Engineer         6864
Architect        6824
Mechanic         6776
Scientist        6744
Accountant       6744
Developer        6720
Media_Manager    6720
Teacher          6672
Entrepreneur     6648
Doctor           6568
Journalist       6536
Manager          6432
Musician         6352
Writer           6304
Name: count, dtype: int64

## Type_of_Loan

### Train Set

In [28]:
df_train.groupby('Customer_ID')['Type_of_Loan'].value_counts(dropna=False)

Customer_ID  Type_of_Loan                                                                                         
1006         Credit-Builder Loan, and Payday Loan                                                                     8
1007         Home Equity Loan, Mortgage Loan, and Student Loan                                                        8
1008         NaN                                                                                                      8
1009         Credit-Builder Loan, Student Loan, Not Specified, and Student Loan                                       8
1011         Personal Loan, Auto Loan, and Auto Loan                                                                  8
                                                                                                                     ..
50984        Home Equity Loan, Mortgage Loan, Payday Loan, Mortgage Loan, Mortgage Loan, and Payday Loan              8
50990        Mortgage Loan, Auto Loan, and Au

In [29]:
df_train['Type_of_Loan'].replace([np.NaN], 'No Data', inplace=True)

In [30]:
df_train.groupby('Customer_ID')['Type_of_Loan'].value_counts(dropna=False)

Customer_ID  Type_of_Loan                                                                                         
1006         Credit-Builder Loan, and Payday Loan                                                                     8
1007         Home Equity Loan, Mortgage Loan, and Student Loan                                                        8
1008         No Data                                                                                                  8
1009         Credit-Builder Loan, Student Loan, Not Specified, and Student Loan                                       8
1011         Personal Loan, Auto Loan, and Auto Loan                                                                  8
                                                                                                                     ..
50984        Home Equity Loan, Mortgage Loan, Payday Loan, Mortgage Loan, Mortgage Loan, and Payday Loan              8
50990        Mortgage Loan, Auto Loan, and Au

## Credit_Mix

In [31]:
df_train['Credit_Mix'].value_counts(dropna=False)

Credit_Mix
Standard    36479
Good        24337
NaN         20195
Bad         18989
Name: count, dtype: int64

In [32]:
Object_NaN_Values_Reassign_Group_Mode(df_train, 'Customer_ID', 'Credit_Mix')


Before Assigning: Credit_Mix: have 20195 NaN Values

Before Assigning Example Credit_Mix:

['Standard', 'Standard', 'Standard', 'Standard', 'Standard', nan, 'Standard', 'Standard']
[nan, 'Standard', 'Standard', 'Standard', nan, 'Standard', nan, 'Standard']
[nan, 'Standard', 'Standard', nan, 'Standard', 'Standard', 'Standard', 'Standard']
['Standard', nan, 'Standard', 'Standard', 'Standard', 'Standard', 'Standard', 'Standard']
[nan, nan, 'Standard', 'Standard', 'Standard', 'Standard', 'Standard', 'Standard']

After Assigning Example Credit_Mix:

['Standard', 'Standard', 'Standard', 'Standard', 'Standard', 'Standard', 'Standard', 'Standard']
['Standard', 'Standard', 'Standard', 'Standard', 'Standard', 'Standard', 'Standard', 'Standard']
['Standard', 'Standard', 'Standard', 'Standard', 'Standard', 'Standard', 'Standard', 'Standard']
['Standard', 'Standard', 'Standard', 'Standard', 'Standard', 'Standard', 'Standard', 'Standard']
['Standard', 'Standard', 'Standard', 'Standard', 'Standard',

In [33]:
df_train['Credit_Mix'].value_counts(dropna=False)

Credit_Mix
Standard    45848
Good        30384
Bad         23768
Name: count, dtype: int64

## Payment_of_Min_Amount

In [34]:
df_train['Payment_of_Min_Amount'].value_counts(dropna=False)

Payment_of_Min_Amount
Yes    52326
No     35667
NM     12007
Name: count, dtype: int64

## Payment_Behaviour

In [35]:
df_train['Payment_Behaviour'].value_counts(dropna=False)

Payment_Behaviour
Low_spent_Small_value_payments      25513
High_spent_Medium_value_payments    17540
Low_spent_Medium_value_payments     13861
High_spent_Large_value_payments     13721
High_spent_Small_value_payments     11340
Low_spent_Large_value_payments      10425
NaN                                  7600
Name: count, dtype: int64

In [36]:
Object_NaN_Values_Reassign_Group_Mode(df_train, 'Customer_ID', 'Payment_Behaviour')


Before Assigning: Payment_Behaviour: have 7600 NaN Values

Before Assigning Example Payment_Behaviour:

['High_spent_Medium_value_payments', 'Low_spent_Medium_value_payments', 'Low_spent_Small_value_payments', 'High_spent_Small_value_payments', 'Low_spent_Large_value_payments', nan, 'Low_spent_Large_value_payments', 'Low_spent_Small_value_payments']
['High_spent_Medium_value_payments', 'High_spent_Medium_value_payments', 'Low_spent_Small_value_payments', 'High_spent_Medium_value_payments', 'Low_spent_Small_value_payments', nan, 'Low_spent_Small_value_payments', 'Low_spent_Small_value_payments']
['High_spent_Small_value_payments', 'Low_spent_Large_value_payments', 'Low_spent_Small_value_payments', 'High_spent_Small_value_payments', 'High_spent_Medium_value_payments', 'High_spent_Small_value_payments', 'Low_spent_Large_value_payments', 'Low_spent_Large_value_payments']
['High_spent_Medium_value_payments', 'High_spent_Large_value_payments', 'High_spent_Medium_value_payments', 'Low_spent_

In [37]:
df_train['Payment_Behaviour'].value_counts(dropna=False)

Payment_Behaviour
Low_spent_Small_value_payments      27767
High_spent_Medium_value_payments    19366
High_spent_Large_value_payments     15348
Low_spent_Medium_value_payments     14621
High_spent_Small_value_payments     11980
Low_spent_Large_value_payments      10918
Name: count, dtype: int64

## Fill NaN Values of Numeric Column Function (Outlier)

In [38]:
def get_iqr_lower_upper(df, column, multiply=1.5):
    q1 = df[column].quantile(0.25)
    q3 = df[column].quantile(0.75)
    iqr = q3 -q1
    
    lower = q1-iqr*multiply
    upper = q3+iqr*multiply
    affect = df.loc[(df[column]<lower)|(df[column]>upper)].shape
    print('Outliers:', affect)
    return lower, upper

In [39]:
def Numeric_Wrong_Values_Reassign_Group_Min_Max(df, groupby, column, inplace=True):      

    def get_group_min_max(df, groupby, column):            
        cur = df[df[column].notna()].groupby(groupby)[column].apply(list)
        x, y = cur.apply(lambda x: stats.mode(x, keepdims=True)).apply([min, max])
        return x[0][0], y[0][0]
    
    def make_group_NaN_and_fill_mode(df, groupby, column, inplace=True):
        df_dropped = df[df[column].notna()].groupby(groupby)[column].apply(list)
        x, y = df_dropped.apply(lambda x: stats.mode(x, keepdims=True)).apply([min, max])
        mini, maxi = x[0][0], y[0][0]
        
        col = df[column].apply(lambda x: np.NaN if ((x < mini) | (x > maxi) | (x < 0)) else x)
        
        mode_by_group = df.groupby(groupby)[column].transform(lambda x: x.mode()[0] if not x.mode().empty else np.NaN)
        result = col.fillna(mode_by_group)
        
        if inplace:
            df[column] = result
        else:
            return result
            
    if inplace:   
        if df[column].isna().sum():
            nan_count = df[column].isna().sum()
            print(f'\nBefore Assigning: {column}: {nan_count} NaN Values\n')
        
        print("\nExisting Min, Max Values:")
        print(df[column].apply([min, max]), end='\n') 
        
        mini, maxi = get_group_min_max(df, groupby, column)        
        print(f"\nGroupby {groupby}'s Actual min, max Values:\nmin:\t{mini},\nmax:\t{maxi}\n")        
        
        print(f'\nBefore Assigning Example {column}:\n', *df.groupby(groupby)[column].apply(list).head().values, sep='\n')
        
        make_group_NaN_and_fill_mode(df, groupby, column, inplace)
        
        if df[column].isna().sum():
            nan_count_after = df[column].isna().sum()
            print(f'\nAfter Assigning: {column}: {nan_count_after} NaN Values\n')
        
        print(f'\nAfter Assigning Example {column}:\n', *df.groupby(groupby)[column].apply(list).head().values, sep='\n')
    else:   
        return make_group_NaN_and_fill_mode(df, groupby, column, inplace)

## Month

In [40]:
df_train['Month'].value_counts()

Month
1    12500
2    12500
3    12500
4    12500
5    12500
6    12500
7    12500
8    12500
Name: count, dtype: int64

## Age

In [41]:
Numeric_Wrong_Values_Reassign_Group_Min_Max(df_train, 'Customer_ID', 'Age')


Existing Min, Max Values:
min    -500
max    8698
Name: Age, dtype: int32

Groupby Customer_ID's Actual min, max Values:
min:	14,
max:	56


Before Assigning Example Age:

[37, 38, 38, 8153, 38, 38, 38, 38]
[48, 48, 48, 48, 48, 48, 48, 48]
[3452, 37, 37, 37, 37, 37, 37, 37]
[22, 22, 22, 22, 22, 22, 22, 23]
[43, 44, 44, 44, 44, 44, 44, 44]

After Assigning Example Age:

[37.0, 38.0, 38.0, 38.0, 38.0, 38.0, 38.0, 38.0]
[48.0, 48.0, 48.0, 48.0, 48.0, 48.0, 48.0, 48.0]
[37.0, 37.0, 37.0, 37.0, 37.0, 37.0, 37.0, 37.0]
[22.0, 22.0, 22.0, 22.0, 22.0, 22.0, 22.0, 23.0]
[43.0, 44.0, 44.0, 44.0, 44.0, 44.0, 44.0, 44.0]


## SSN

In [42]:
df_train.SSN.value_counts(dropna=False)

SSN
NaN             5572
78735990.00        8
486783816.00       8
750677525.00       8
903500305.00       8
                ... 
856066147.00       4
753722651.00       4
331281921.00       4
604626133.00       4
286449634.00       4
Name: count, Length: 12501, dtype: int64

In [43]:
Numeric_Wrong_Values_Reassign_Group_Min_Max(df_train, 'Customer_ID', 'SSN')


Before Assigning: SSN: 5572 NaN Values


Existing Min, Max Values:
min       81349.00
max   999993421.00
Name: SSN, dtype: float64

Groupby Customer_ID's Actual min, max Values:
min:	81349.0,
max:	999993421.0


Before Assigning Example SSN:

[354656948.0, 354656948.0, 354656948.0, 354656948.0, 354656948.0, 354656948.0, 354656948.0, 354656948.0]
[964812710.0, 964812710.0, 964812710.0, 964812710.0, 964812710.0, 964812710.0, 964812710.0, 964812710.0]
[802194704.0, nan, 802194704.0, 802194704.0, 802194704.0, 802194704.0, 802194704.0, 802194704.0]
[nan, nan, 891062189.0, 891062189.0, 891062189.0, 891062189.0, 891062189.0, 891062189.0]
[422130011.0, 422130011.0, 422130011.0, 422130011.0, 422130011.0, 422130011.0, 422130011.0, 422130011.0]

After Assigning Example SSN:

[354656948.0, 354656948.0, 354656948.0, 354656948.0, 354656948.0, 354656948.0, 354656948.0, 354656948.0]
[964812710.0, 964812710.0, 964812710.0, 964812710.0, 964812710.0, 964812710.0, 964812710.0, 964812710.0]
[802194704.0, 8

## Annual_Income

In [44]:
df_train.Annual_Income.value_counts(dropna=False)

Annual_Income
17816.75       16
22434.16       16
40341.16       16
17273.83       16
109945.32      16
               ..
17079092.00     1
1910572.00      1
20179076.00     1
7980216.00      1
8299495.00      1
Name: count, Length: 13487, dtype: int64

In [45]:
Numeric_Wrong_Values_Reassign_Group_Min_Max(df_train, 'Customer_ID', 'Annual_Income')


Existing Min, Max Values:
min       7005.93
max   24198062.00
Name: Annual_Income, dtype: float64

Groupby Customer_ID's Actual min, max Values:
min:	7005.93,
max:	179987.28


Before Assigning Example Annual_Income:

[16756.18, 16756.18, 16756.18, 16756.18, 16756.18, 16756.18, 16756.18, 16756.18]
[21212.91, 21212.91, 21212.91, 21212.91, 21212.91, 21212.91, 21212.91, 21212.91]
[33540.43, 33540.43, 33540.43, 33540.43, 33540.43, 33540.43, 33540.43, 33540.43]
[80983.64, 80983.64, 80983.64, 80983.64, 80983.64, 80983.64, 80983.64, 80983.64]
[104142.56, 104142.56, 104142.56, 104142.56, 104142.56, 104142.56, 104142.56, 104142.56]

After Assigning Example Annual_Income:

[16756.18, 16756.18, 16756.18, 16756.18, 16756.18, 16756.18, 16756.18, 16756.18]
[21212.91, 21212.91, 21212.91, 21212.91, 21212.91, 21212.91, 21212.91, 21212.91]
[33540.43, 33540.43, 33540.43, 33540.43, 33540.43, 33540.43, 33540.43, 33540.43]
[80983.64, 80983.64, 80983.64, 80983.64, 80983.64, 80983.64, 80983.64, 80983.64]
[104

## Monthly_Inhand_Salary

In [46]:
df_train.Monthly_Inhand_Salary.value_counts(dropna=False)

Monthly_Inhand_Salary
NaN        15002
2295.06       15
6082.19       15
6769.13       15
6358.96       15
           ...  
1087.55        1
3189.21        1
5640.12        1
7727.56        1
2443.65        1
Name: count, Length: 13236, dtype: int64

In [47]:
Numeric_Wrong_Values_Reassign_Group_Min_Max(df_train, 'Customer_ID', 'Monthly_Inhand_Salary')


Before Assigning: Monthly_Inhand_Salary: 15002 NaN Values


Existing Min, Max Values:
min     303.65
max   15204.63
Name: Monthly_Inhand_Salary, dtype: float64

Groupby Customer_ID's Actual min, max Values:
min:	303.6454166666666,
max:	15204.633333333331


Before Assigning Example Monthly_Inhand_Salary:

[1331.3483333333334, 1331.3483333333334, 1331.3483333333334, 1331.3483333333334, 1331.3483333333334, 1331.3483333333334, 1331.3483333333334, 1331.3483333333334]
[1496.7425, 1496.7425, 1496.7425, 1496.7425, nan, 1496.7425, 1496.7425, 1496.7425]
[2655.035833333333, 2655.035833333333, 2655.035833333333, 2655.035833333333, 2655.035833333333, 2655.035833333333, 2655.035833333333, 2655.035833333333]
[6692.636666666666, 6692.636666666666, 6692.636666666666, 6692.636666666666, 6692.636666666666, 6692.636666666666, 6692.636666666666, 6692.636666666666]
[8433.546666666667, 8433.546666666667, 8433.546666666667, 8433.546666666667, 8433.546666666667, 8433.546666666667, 8433.546666666667, nan]

Aft

## Num_Bank_Accounts

In [48]:
df_train.Num_Bank_Accounts.value_counts(dropna=False)

Num_Bank_Accounts
6       13001
7       12823
8       12765
4       12186
5       12118
        ...  
1626        1
1470        1
887         1
211         1
697         1
Name: count, Length: 943, dtype: int64

In [49]:
df_train.loc[df_train['Num_Bank_Accounts']<0, 'Num_Bank_Accounts'] = 0

In [50]:
Numeric_Wrong_Values_Reassign_Group_Min_Max(df_train, 'Customer_ID', 'Num_Bank_Accounts')


Existing Min, Max Values:
min       0
max    1798
Name: Num_Bank_Accounts, dtype: int64

Groupby Customer_ID's Actual min, max Values:
min:	0,
max:	10


Before Assigning Example Num_Bank_Accounts:

[9, 9, 9, 9, 9, 9, 9, 9]
[3, 3, 3, 1174, 3, 3, 3, 3]
[6, 6, 6, 6, 6, 6, 6, 6]
[6, 6, 6, 6, 6, 6, 6, 6]
[3, 3, 3, 3, 3, 3, 3, 3]

After Assigning Example Num_Bank_Accounts:

[9.0, 9.0, 9.0, 9.0, 9.0, 9.0, 9.0, 9.0]
[3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0]
[6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0]
[6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0]
[3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0]


## Num_Credit_Card

In [51]:
df_train.Num_Credit_Card.value_counts(dropna=False)

Num_Credit_Card
5       18459
7       16615
6       16559
4       14030
3       13277
        ...  
791         1
1118        1
657         1
640         1
679         1
Name: count, Length: 1179, dtype: int64

In [52]:
Numeric_Wrong_Values_Reassign_Group_Min_Max(df_train, 'Customer_ID', 'Num_Credit_Card')


Existing Min, Max Values:
min       0
max    1499
Name: Num_Credit_Card, dtype: int64

Groupby Customer_ID's Actual min, max Values:
min:	0,
max:	11


Before Assigning Example Num_Credit_Card:

[6, 6, 6, 6, 6, 6, 6, 6]
[4, 4, 4, 4, 4, 4, 4, 888]
[3, 3, 3, 3, 3, 3, 3, 3]
[3, 3, 3, 3, 725, 3, 3, 3]
[5, 5, 5, 5, 5, 5, 5, 5]

After Assigning Example Num_Credit_Card:

[6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0]
[4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0]
[3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0]
[3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0]
[5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0]


## Interest_Rate

In [53]:
df_train.Interest_Rate.value_counts(dropna=False)

Interest_Rate
8       5012
5       4979
6       4721
12      4540
10      4540
        ... 
4995       1
1899       1
2120       1
5762       1
5729       1
Name: count, Length: 1750, dtype: int64

In [54]:
Numeric_Wrong_Values_Reassign_Group_Min_Max(df_train, 'Customer_ID', 'Interest_Rate')


Existing Min, Max Values:
min       1
max    5797
Name: Interest_Rate, dtype: int64

Groupby Customer_ID's Actual min, max Values:
min:	1,
max:	34


Before Assigning Example Interest_Rate:

[22, 22, 22, 22, 22, 22, 22, 22]
[10, 10, 10, 10, 10, 10, 10, 10]
[17, 17, 17, 17, 17, 17, 17, 17]
[15, 15, 15, 15, 15, 15, 15, 15]
[5, 5, 5, 5, 5, 5, 5, 5]

After Assigning Example Interest_Rate:

[22.0, 22.0, 22.0, 22.0, 22.0, 22.0, 22.0, 22.0]
[10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0]
[17.0, 17.0, 17.0, 17.0, 17.0, 17.0, 17.0, 17.0]
[15.0, 15.0, 15.0, 15.0, 15.0, 15.0, 15.0, 15.0]
[5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0]


## Num_of_Loan

In [55]:
df_train.Num_of_Loan.value_counts(dropna=False)

Num_of_Loan
3       15104
2       15032
4       14743
0       10930
1       10606
        ...  
119         1
321         1
1439        1
663         1
966         1
Name: count, Length: 414, dtype: int64

In [56]:
Numeric_Wrong_Values_Reassign_Group_Min_Max(df_train, 'Customer_ID', 'Num_of_Loan')


Existing Min, Max Values:
min    -100
max    1496
Name: Num_of_Loan, dtype: int32

Groupby Customer_ID's Actual min, max Values:
min:	0,
max:	9


Before Assigning Example Num_of_Loan:

[2, 2, 2, 2, 2, 2, 2, 2]
[3, 3, 3, 3, -100, 3, 3, 3]
[0, 0, 0, 0, 0, 0, 0, 0]
[4, 4, 4, 4, 4, 4, 4, 4]
[3, 3, 3, 3, 3, 3, 3, 3]

After Assigning Example Num_of_Loan:

[2.0, 2.0, 2.0, 2.0, 2.0, 2.0, 2.0, 2.0]
[3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0]
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
[4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0]
[3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0]


## Delay_from_due_date

In [57]:
df_train.Delay_from_due_date.value_counts(dropna=False)

Delay_from_due_date
 15    3596
 13    3424
 8     3324
 14    3313
 10    3281
       ... 
-4       62
 65      56
-5       33
 66      32
 67      22
Name: count, Length: 73, dtype: int64

In [58]:
df_train.loc[df_train['Delay_from_due_date']<0, 'Delay_from_due_date'] = None

In [59]:
Numeric_Wrong_Values_Reassign_Group_Min_Max(df_train, 'Customer_ID', 'Delay_from_due_date')


Before Assigning: Delay_from_due_date: 591 NaN Values


Existing Min, Max Values:
min    0.00
max   67.00
Name: Delay_from_due_date, dtype: float64

Groupby Customer_ID's Actual min, max Values:
min:	0.0,
max:	62.0


Before Assigning Example Delay_from_due_date:

[48.0, 48.0, 48.0, 48.0, 48.0, 48.0, 48.0, 48.0]
[19.0, 19.0, 19.0, 19.0, 19.0, 19.0, 19.0, 16.0]
[25.0, 28.0, 22.0, 26.0, 26.0, 26.0, 26.0, 26.0]
[19.0, 19.0, 19.0, 19.0, 19.0, 19.0, 19.0, 19.0]
[15.0, 15.0, 20.0, 16.0, 20.0, 20.0, 20.0, 24.0]

After Assigning Example Delay_from_due_date:

[48.0, 48.0, 48.0, 48.0, 48.0, 48.0, 48.0, 48.0]
[19.0, 19.0, 19.0, 19.0, 19.0, 19.0, 19.0, 16.0]
[25.0, 28.0, 22.0, 26.0, 26.0, 26.0, 26.0, 26.0]
[19.0, 19.0, 19.0, 19.0, 19.0, 19.0, 19.0, 19.0]
[15.0, 15.0, 20.0, 16.0, 20.0, 20.0, 20.0, 24.0]


## Num_of_Delayed_Payment

In [60]:
df_train.Num_of_Delayed_Payment.value_counts(dropna=False)

Num_of_Delayed_Payment
NaN        7002
19.00      5481
17.00      5412
16.00      5312
10.00      5309
           ... 
1534.00       1
3739.00       1
3313.00       1
4191.00       1
2047.00       1
Name: count, Length: 712, dtype: int64

In [61]:
df_train.loc[df_train['Num_of_Delayed_Payment']<0, 'Num_of_Delayed_Payment'] = None

In [62]:
Numeric_Wrong_Values_Reassign_Group_Min_Max(df_train, 'Customer_ID', 'Num_of_Delayed_Payment')


Before Assigning: Num_of_Delayed_Payment: 7646 NaN Values


Existing Min, Max Values:
min      0.00
max   4397.00
Name: Num_of_Delayed_Payment, dtype: float64

Groupby Customer_ID's Actual min, max Values:
min:	0.0,
max:	28.0


Before Assigning Example Num_of_Delayed_Payment:

[10.0, 12.0, 12.0, 13.0, nan, 12.0, 11.0, 12.0]
[19.0, 19.0, 19.0, 19.0, 19.0, 21.0, 20.0, 19.0]
[11.0, 11.0, 11.0, nan, 11.0, nan, 11.0, 13.0]
[18.0, 18.0, 18.0, 18.0, 18.0, nan, 20.0, 18.0]
[17.0, 16.0, 14.0, nan, 17.0, 14.0, 11.0, 14.0]

After Assigning Example Num_of_Delayed_Payment:

[10.0, 12.0, 12.0, 13.0, 12.0, 12.0, 11.0, 12.0]
[19.0, 19.0, 19.0, 19.0, 19.0, 21.0, 20.0, 19.0]
[11.0, 11.0, 11.0, 11.0, 11.0, 11.0, 11.0, 13.0]
[18.0, 18.0, 18.0, 18.0, 18.0, 18.0, 20.0, 18.0]
[17.0, 16.0, 14.0, 14.0, 17.0, 14.0, 11.0, 14.0]


In [63]:
df_train.Num_of_Delayed_Payment.value_counts(dropna=False)

Num_of_Delayed_Payment
19.00    5967
17.00    5831
10.00    5813
16.00    5780
15.00    5711
18.00    5669
20.00    5570
12.00    5504
9.00     5410
8.00     5294
11.00    5259
14.00    4502
13.00    4335
21.00    2723
7.00     2537
6.00     2493
22.00    2491
5.00     2267
23.00    2183
0.00     2086
3.00     2075
2.00     2073
1.00     2045
4.00     1983
25.00    1853
24.00    1843
26.00     322
27.00     250
28.00     131
Name: count, dtype: int64

## Changed_Credit_Limit

In [64]:
df_train.Changed_Credit_Limit.value_counts(dropna=False)

Changed_Credit_Limit
NaN      2091
8.22      133
11.50     127
11.32     126
7.35      121
         ... 
-1.84       1
0.89        1
28.06       1
1.56        1
21.17       1
Name: count, Length: 4384, dtype: int64

In [65]:
Numeric_Wrong_Values_Reassign_Group_Min_Max(df_train, 'Customer_ID', 'Changed_Credit_Limit')


Before Assigning: Changed_Credit_Limit: 2091 NaN Values


Existing Min, Max Values:
min   -6.49
max   36.97
Name: Changed_Credit_Limit, dtype: float64

Groupby Customer_ID's Actual min, max Values:
min:	-5.01,
max:	29.98


Before Assigning Example Changed_Credit_Limit:

[10.66, 10.66, 10.66, 10.66, 10.66, 10.66, 10.66, 10.66]
[12.13, 5.13, 5.13, 5.13, 5.13, 2.13, 5.13, 5.13]
[14.11, 14.11, 14.11, 14.11, 14.11, 14.11, 14.11, 14.11]
[16.91, 16.91, 16.91, 16.91, 16.91, 16.91, 19.91, 16.91]
[15.28, 15.28, 15.28, 15.28, 19.28, 15.28, 15.28, 15.28]

After Assigning Example Changed_Credit_Limit:

[10.66, 10.66, 10.66, 10.66, 10.66, 10.66, 10.66, 10.66]
[12.13, 5.13, 5.13, 5.13, 5.13, 2.13, 5.13, 5.13]
[14.11, 14.11, 14.11, 14.11, 14.11, 14.11, 14.11, 14.11]
[16.91, 16.91, 16.91, 16.91, 16.91, 16.91, 19.91, 16.91]
[15.28, 15.28, 15.28, 15.28, 19.28, 15.28, 15.28, 15.28]


In [66]:
df_train.Changed_Credit_Limit.value_counts(dropna=False)

Changed_Credit_Limit
8.22     137
11.50    128
11.32    127
10.06    124
7.35     123
        ... 
15.01      1
0.08       1
23.32      1
1.07       1
21.17      1
Name: count, Length: 3532, dtype: int64

## Num_Credit_Inquiries

In [67]:
df_train.Num_Credit_Inquiries.value_counts(dropna=False)

Num_Credit_Inquiries
4.00       11271
3.00        8890
6.00        8111
7.00        8058
2.00        8028
           ...  
1721.00        1
1750.00        1
2397.00        1
621.00         1
74.00          1
Name: count, Length: 1224, dtype: int64

In [68]:
Numeric_Wrong_Values_Reassign_Group_Min_Max(df_train, 'Customer_ID', 'Num_Credit_Inquiries')


Before Assigning: Num_Credit_Inquiries: 1965 NaN Values


Existing Min, Max Values:
min      0.00
max   2597.00
Name: Num_Credit_Inquiries, dtype: float64

Groupby Customer_ID's Actual min, max Values:
min:	0.0,
max:	17.0


Before Assigning Example Num_Credit_Inquiries:

[nan, 8.0, 8.0, 8.0, 8.0, 8.0, 8.0, 8.0]
[1.0, 1.0, 1.0, 1.0, 1196.0, 1.0, 1.0, 1.0]
[6.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0]
[7.0, 7.0, 7.0, 7.0, 7.0, 7.0, 7.0, 7.0]
[6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 692.0]

After Assigning Example Num_Credit_Inquiries:

[8.0, 8.0, 8.0, 8.0, 8.0, 8.0, 8.0, 8.0]
[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
[6.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0]
[7.0, 7.0, 7.0, 7.0, 7.0, 7.0, 7.0, 7.0]
[6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0]


## Outstanding_Debt

In [69]:
df_train.Outstanding_Debt.value_counts(dropna=False)

Outstanding_Debt
1109.03    24
1151.70    24
1360.45    24
460.46     24
1058.13    16
           ..
4230.04     8
641.99      8
98.61       8
2614.48     8
502.38      8
Name: count, Length: 12203, dtype: int64

In [70]:
Numeric_Wrong_Values_Reassign_Group_Min_Max(df_train, 'Customer_ID', 'Outstanding_Debt')


Existing Min, Max Values:
min      0.23
max   4998.07
Name: Outstanding_Debt, dtype: float64

Groupby Customer_ID's Actual min, max Values:
min:	0.23,
max:	4998.07


Before Assigning Example Outstanding_Debt:

[1941.73, 1941.73, 1941.73, 1941.73, 1941.73, 1941.73, 1941.73, 1941.73]
[993.15, 993.15, 993.15, 993.15, 993.15, 993.15, 993.15, 993.15]
[1138.97, 1138.97, 1138.97, 1138.97, 1138.97, 1138.97, 1138.97, 1138.97]
[982.44, 982.44, 982.44, 982.44, 982.44, 982.44, 982.44, 982.44]
[1371.8, 1371.8, 1371.8, 1371.8, 1371.8, 1371.8, 1371.8, 1371.8]

After Assigning Example Outstanding_Debt:

[1941.73, 1941.73, 1941.73, 1941.73, 1941.73, 1941.73, 1941.73, 1941.73]
[993.15, 993.15, 993.15, 993.15, 993.15, 993.15, 993.15, 993.15]
[1138.97, 1138.97, 1138.97, 1138.97, 1138.97, 1138.97, 1138.97, 1138.97]
[982.44, 982.44, 982.44, 982.44, 982.44, 982.44, 982.44, 982.44]
[1371.8, 1371.8, 1371.8, 1371.8, 1371.8, 1371.8, 1371.8, 1371.8]


## Credit_Utilization_Ratio

In [71]:
df_train.Credit_Utilization_Ratio.value_counts(dropna=False)

Credit_Utilization_Ratio
26.82    1
28.33    1
30.02    1
25.48    1
33.93    1
        ..
30.69    1
38.73    1
30.02    1
27.28    1
34.19    1
Name: count, Length: 100000, dtype: int64

In [72]:
df_train.Credit_Utilization_Ratio.isna().sum()

0

## Credit_History_Age

In [73]:
df_train.Credit_History_Age.value_counts(dropna=False)

Credit_History_Age
NaN       9030
191.00     446
232.00     445
233.00     444
215.00     443
          ... 
3.00        20
2.00        15
403.00      14
404.00      12
1.00         2
Name: count, Length: 405, dtype: int64

In [74]:
df_train['Credit_History_Age'] = df_train.groupby('Customer_ID')['Credit_History_Age'].transform(lambda x: x.interpolate().bfill().ffill())

In [75]:
df_train.Credit_History_Age.value_counts(dropna=False)

Credit_History_Age
232.00    488
191.00    486
190.00    483
233.00    483
215.00    483
         ... 
3.00       21
403.00     18
2.00       15
404.00     12
1.00        2
Name: count, Length: 404, dtype: int64

## Total_EMI_per_month

In [76]:
df_train.Total_EMI_per_month.value_counts(dropna=False)

Total_EMI_per_month
0.00        10613
49.57           8
73.53           8
22.96           8
38.66           8
            ...  
36408.00        1
23760.00        1
24612.00        1
24325.00        1
58638.00        1
Name: count, Length: 14950, dtype: int64

In [77]:
Numeric_Wrong_Values_Reassign_Group_Min_Max(df_train, 'Customer_ID', 'Total_EMI_per_month')


Existing Min, Max Values:
min       0.00
max   82331.00
Name: Total_EMI_per_month, dtype: float64

Groupby Customer_ID's Actual min, max Values:
min:	0.0,
max:	1779.1032538262775


Before Assigning Example Total_EMI_per_month:

[27.44208910654816, 27.44208910654816, 27.44208910654816, 27.44208910654816, 27.44208910654816, 27.44208910654816, 27.44208910654816, 32972.0]
[45.74570037068675, 45.74570037068675, 45.74570037068675, 45.74570037068675, 45.74570037068675, 45.74570037068675, 45.74570037068675, 45.74570037068675]
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
[188.45659522353543, 188.45659522353543, 188.45659522353543, 188.45659522353543, 188.45659522353543, 188.45659522353543, 188.45659522353543, 188.45659522353543]
[257.7386460249556, 257.7386460249556, 257.7386460249556, 257.7386460249556, 257.7386460249556, 257.7386460249556, 257.7386460249556, 257.7386460249556]

After Assigning Example Total_EMI_per_month:

[27.44208910654816, 27.44208910654816, 27.44208910654816, 27.442089106548

## Amount_invested_monthly

In [78]:
df_train.Amount_invested_monthly.value_counts(dropna=False)

Amount_invested_monthly
NaN         4479
10000.00    4305
0.00         169
36.66          1
89.74          1
            ... 
36.54          1
93.45          1
140.81         1
38.74          1
167.16         1
Name: count, Length: 91050, dtype: int64

In [79]:
df_train.loc[df_train['Amount_invested_monthly']>=10000, 'Amount_invested_monthly'] = None

In [80]:
Numeric_Wrong_Values_Reassign_Group_Min_Max(df_train, 'Customer_ID', 'Amount_invested_monthly')


Before Assigning: Amount_invested_monthly: 8784 NaN Values


Existing Min, Max Values:
min      0.00
max   1977.33
Name: Amount_invested_monthly, dtype: float64

Groupby Customer_ID's Actual min, max Values:
min:	0.0,
max:	510.7262372637133


Before Assigning Example Amount_invested_monthly:

[45.30106826949194, 90.07842318605292, 61.73271462991577, 56.4949816634165, 51.726244263612, 60.82828780912217, 95.6486475869488, 66.71824831147686]
[30.373471914127194, 44.31995483866171, nan, 62.81248610216079, 148.30956484525, nan, 59.30896122323683, 118.54244614778156]
[118.8806978910128, nan, 337.1997406214716, 170.8689598433119, 74.1980692925824, 105.4573791889266, 197.85037247610126, 165.20443547590202]
[149.37725143584314, 146.97809602243544, 101.12020124815511, 492.3974911743151, 264.1791118544216, 261.69017894140734, 200.64143588981403, 126.07161584554541]
[292.2127037381353, nan, 187.59489650806347, 275.635709774446, 487.8062062008502, nan, 124.29952362711306, 481.7990883470461]

After

In [81]:
df_train.Amount_invested_monthly.value_counts(dropna=False)

Amount_invested_monthly
0.00      262
93.73       8
186.93      8
329.46      8
337.93      8
         ... 
174.41      1
83.06       1
74.16       1
80.32       1
167.16      1
Name: count, Length: 84191, dtype: int64

## Monthly_Balance

In [82]:
df_train.Monthly_Balance.value_counts(dropna=False)

Monthly_Balance
NaN                                1200
-333333333333333314856026112.00       9
312.49                                1
347.41                                1
254.97                                1
                                   ... 
366.29                                1
151.19                                1
306.75                                1
278.87                                1
393.67                                1
Name: count, Length: 98793, dtype: int64

In [83]:
df_train.loc[df_train['Monthly_Balance']<0, 'Monthly_Balance'] = None

In [84]:
Numeric_Wrong_Values_Reassign_Group_Min_Max(df_train, 'Customer_ID', 'Monthly_Balance')


Before Assigning: Monthly_Balance: 1209 NaN Values


Existing Min, Max Values:
min      0.01
max   1602.04
Name: Monthly_Balance, dtype: float64

Groupby Customer_ID's Actual min, max Values:
min:	0.007759664775335295,
max:	1183.9306960885192


Before Assigning Example Monthly_Balance:

[310.39167595729333, 295.61432104073225, 333.9600295968694, 309.1977625633686, 323.9664999631732, 334.864456417663, 280.04409663983637, 328.97449591530835]
[323.5550777151861, 309.6085947906515, 265.4874646531188, 291.11606352715245, 245.61898478406326, 239.46481499922072, 334.61958840607645, 275.3861034815317]
[406.62288544232047, 350.39351957834924, 218.30384271186168, 354.63462349002145, 441.30551404075095, 420.04620414440666, 337.6532108572321, 370.2991478574313]
[581.4298200072883, 573.8289754206959, 629.6868701949761, 258.40958026881617, 496.6279595887097, 489.116892501724, 560.1656355533173, 594.7354555975859]
[553.4033169035758, 228.71773015063442, 648.0211241336477, 579.9803108672652, 377.8098

In [85]:
df_train.Monthly_Balance.value_counts(dropna=False)

Monthly_Balance
1183.93    8
1014.17    8
372.87     8
236.24     8
1040.21    7
          ..
351.99     1
271.95     1
311.15     1
338.56     1
393.67     1
Name: count, Length: 98068, dtype: int64

## Finish Cleaning

In [86]:
df_train

,ID,Customer_ID,Month,Name,Age,SSN,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,...,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,Credit_Score
0,5634,3392,1,Aaron Maashoh,23.00,821000265.00,Scientist,19114.12,1824.84,3.00,...,Good,809.98,26.82,265.00,No,49.57,80.42,High_spent_Small_value_payments,312.49,Good
1,5635,3392,2,Aaron Maashoh,23.00,821000265.00,Scientist,19114.12,1824.84,3.00,...,Good,809.98,31.94,266.00,No,49.57,118.28,Low_spent_Large_value_payments,284.63,Good
2,5636,3392,3,Aaron Maashoh,23.00,821000265.00,Scientist,19114.12,1824.84,3.00,...,Good,809.98,28.61,267.00,No,49.57,81.70,Low_spent_Medium_value_payments,331.21,Good
3,5637,3392,4,Aaron Maashoh,23.00,821000265.00,Scientist,19114.12,1824.84,3.00,...,Good,809.98,31.38,268.00,No,49.57,199.46,Low_spent_Small_value_payments,223.45,Good
4,5638,3392,5,Aaron Maashoh,23.00,821000265.00,Scientist,19114.12,1824.84,3.00,...,Good,809.98,24.80,269.00,No,49.57,41.42,High_spent_Medium_value_payments,341.49,Good
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,155625,37932,4,Nicks,25.00,78735990.00,Mechanic,39628.99,3359.42,4.00,...,Good,502.38,34.66,378.00,No,35.10,60.97,High_spent_Large_value_payments,479.87,Poor
99996,155626,37932,5,Nicks,25.00,78735990.00,Mechanic,39628.99,3359.42,4.00,...,Good,502.38,40.57,379.00,No,35.10,54.19,High_spent_Medium_value_payments,496.65,Poor
99997,155627,37932,6,Nicks,25.00,78735990.00,Mechanic,39628.99,3359.42,4.00,...,Good,502.38,41.26,380.00,No,35.10,24.03,High_spent_Large_value_payments,516.81,Poor
99998,155628,37932,7,Nicks,25.00,78735990.00,Mechanic,39628.99,3359.42,4.00,...,Good,502.38,33.64,381.00,No,35.10,251.67,Low_spent_Large_value_payments,319.16,Standard


In [87]:
df_train.isna().sum()

ID                          0
Customer_ID                 0
Month                       0
Name                        0
Age                         0
SSN                         0
Occupation                  0
Annual_Income               0
Monthly_Inhand_Salary       0
Num_Bank_Accounts           0
Num_Credit_Card             0
Interest_Rate               0
Num_of_Loan                 0
Type_of_Loan                0
Delay_from_due_date         0
Num_of_Delayed_Payment      0
Changed_Credit_Limit        0
Num_Credit_Inquiries        0
Credit_Mix                  0
Outstanding_Debt            0
Credit_Utilization_Ratio    0
Credit_History_Age          0
Payment_of_Min_Amount       0
Total_EMI_per_month         0
Amount_invested_monthly     0
Payment_Behaviour           0
Monthly_Balance             0
Credit_Score                0
dtype: int64

In [88]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 28 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   ID                        100000 non-null  int64  
 1   Customer_ID               100000 non-null  int64  
 2   Month                     100000 non-null  int32  
 3   Name                      100000 non-null  object 
 4   Age                       100000 non-null  float64
 5   SSN                       100000 non-null  float64
 6   Occupation                100000 non-null  object 
 7   Annual_Income             100000 non-null  float64
 8   Monthly_Inhand_Salary     100000 non-null  float64
 9   Num_Bank_Accounts         100000 non-null  float64
 10  Num_Credit_Card           100000 non-null  float64
 11  Interest_Rate             100000 non-null  float64
 12  Num_of_Loan               100000 non-null  float64
 13  Type_of_Loan              100000 non-null  ob

In [89]:
df_train.describe().T

,count,mean,std,min,25%,50%,75%,max
ID,100000.00,80631.50,43301.49,5634.00,43132.75,80631.50,118130.25,155629.00
Customer_ID,100000.00,25982.67,14340.54,1006.00,13664.50,25777.00,38385.00,50999.00
Month,100000.00,4.50,2.29,1.00,2.75,4.50,6.25,8.00
Age,100000.00,33.31,10.76,14.00,24.00,33.00,42.00,56.00
SSN,100000.00,500461680.26,290826734.39,81349.00,245168577.25,500688611.50,756002666.25,999993421.00
Annual_Income,100000.00,50505.12,38299.42,7005.93,19342.97,36999.71,71683.47,179987.28
Monthly_Inhand_Salary,100000.00,4198.35,3187.40,303.65,1626.76,3095.98,5961.64,15204.63
Num_Bank_Accounts,100000.00,5.37,2.59,0.00,3.00,5.00,7.00,10.00
Num_Credit_Card,100000.00,5.53,2.07,0.00,4.00,5.00,7.00,11.00
Interest_Rate,100000.00,14.53,8.74,1.00,7.00,13.00,20.00,34.00


# Save Cleaned Data

In [90]:
df_train.to_csv("cleaned data/clean_train.csv", index=False)